In [6]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_PATH = Path("Base.csv")
MODEL_PATH = Path("fraud_model.joblib")

In [7]:
# Load Data, Filter month 0–3, & Split Data (month 0–2 Train, month 3 Validasi)
df = pd.read_csv(DATA_PATH)

# Filter varian Base hanya bulan 0 sampai 3
df_sub = df[df["month"].between(0, 3)].copy()

# Pembagian temporal (out-of-time): train = bulan 0-2, validasi = bulan 3
train_df = df_sub[df_sub["month"].isin([0, 1, 2])].copy()
val_df = df_sub[df_sub["month"] == 3].copy()

TARGET = "fraud_bool"
DROP_COLS = [TARGET, "month"]

X_train = train_df.drop(columns=DROP_COLS)
y_train = train_df[TARGET].astype(int)

X_val = val_df.drop(columns=DROP_COLS)
y_val = val_df[TARGET].astype(int)

print(f"Train (month 0-2): {len(X_train):,} baris | Fraud rate: {y_train.mean()*100:.2f}%")
print(f"Val   (month 3)  : {len(X_val):,} baris | Fraud rate: {y_val.mean()*100:.2f}%")

Train (month 0-2): 397,039 baris | Fraud rate: 0.98%
Val   (month 3)  : 150,936 baris | Fraud rate: 0.92%


In [8]:
# Pipeline Preprocessing + LightGBM & Latih Model
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ],
    remainder="drop",
)

model = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

clf_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", model),
    ]
)

clf_pipeline.fit(X_train, y_train)
print("Model berhasil dilatih.")

Model berhasil dilatih.


In [9]:
# Evaluasi: Hitung TPR pada FPR ≤ 5% di Data Validasi
val_probs = clf_pipeline.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, val_probs)

fpr, tpr, thresholds = roc_curve(y_val, val_probs)

# Ambil semua titik threshold dengan FPR <= 5% (0.05)
valid_idx = np.where(fpr <= 0.05)[0]

# Pilih indeks dengan TPR tertinggi pada batas FPR <= 5%
best_idx = valid_idx[np.argmax(tpr[valid_idx])]

best_threshold = float(thresholds[best_idx])
val_fpr_at_5 = float(fpr[best_idx])
val_tpr_at_5 = float(tpr[best_idx])

print(f"ROC-AUC (Validation)       : {roc_auc:.4f}")
print(f"Threshold (FPR <= 5%)      : {best_threshold:.6f}")
print(f"FPR Aktual (Validation)    : {val_fpr_at_5 * 100:.2f}%")
print(f"TPR pada FPR <= 5% (Recall): {val_tpr_at_5 * 100:.2f}%")

ROC-AUC (Validation)       : 0.9097
Threshold (FPR <= 5%)      : 0.807555
FPR Aktual (Validation)    : 4.92%
TPR pada FPR <= 5% (Recall): 55.82%


In [10]:
# Simpan Model & Uji Muat Ulang
artifact = {
    "pipeline": clf_pipeline,
    "threshold": best_threshold,
    "val_roc_auc": float(roc_auc),
    "val_fpr": val_fpr_at_5,
    "val_tpr": val_tpr_at_5,
}

joblib.dump(artifact, MODEL_PATH)
print(f"Model tersimpan di: {MODEL_PATH.resolve()}")

# Uji muat ulang dari disk
loaded = joblib.load(MODEL_PATH)
sample_probs = loaded["pipeline"].predict_proba(X_val.iloc[:5])[:, 1]
sample_preds = (sample_probs >= loaded["threshold"]).astype(int)

print("Skor probabilitas 5 data pertama:", np.round(sample_probs, 4))
print("Prediksi label (0/1)            :", sample_preds)

Model tersimpan di: /content/fraud_model.joblib
Skor probabilitas 5 data pertama: [0.9616 0.0026 0.0128 0.0453 0.2359]
Prediksi label (0/1)            : [1 0 0 0 0]
